# QC-large four-model comparison

This notebook focuses on the QC-large strict-left dataset because it is the only dataset in this repo
that contains the exact four-model comparison:

- BrainODE PCA150
- PCA150 cocycle flow
- SIREN cocycle flow
- SIREN latent ODE

The figures stay on the current evaluated outputs and only decode new meshes when you explicitly run
the anchor forecast cells.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root.parent == root:
        raise RuntimeError("Could not locate repo root from current working directory.")
    root = root.parent

script_dir = root / "examples" / "ADNI_1_L_No_MCI" / "brainode_comparison_task3_core_brainode_original" / "scripts"
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

import longitudinal_visual_notebook_support as lv
lv = importlib.reload(lv)

ctx = lv.create_context(device="auto")
display(Markdown(
    "Using `longitudinal_visual_notebook_support.py`. "
    "Observed-pair plots are light. Anchor forecast cells decode meshes for the SIREN models and run best on CUDA."
))


Using `longitudinal_visual_notebook_support.py`. Observed-pair plots are light. Anchor forecast cells decode meshes for the SIREN models and run best on CUDA.

## Ground-truth CN vs AD aging

Relative change is measured from each subject's baseline scan. The shaded band is the interquartile range
across subjects with at least two scans.


In [2]:
display(Markdown(
    "### Ground-truth relative aging curves\n"
    "- **Input:** all QC-large ground-truth scan volumes in `train`, `val`, and `test`.\n"
    "- **Output:** mean relative hippocampus volume change from each subject's own baseline, shown separately for `CN` and `AD`.\n"
    "- **Calculation:** for each subject, `relative change = 100 * (current volume - baseline volume) / baseline volume`."
))
fig = ctx.plot_ground_truth_cn_ad_aging()
fig.show()
display(Markdown(
    "After this graph: more negative values indicate stronger volume loss from baseline. "
    "Because the axis is baseline-relative, subjects with different absolute hippocampus sizes are directly comparable."
))

display(Markdown(
    "### Ground-truth annualized start-to-end change\n"
    "- **Input:** the first and last observed scan for each subject.\n"
    "- **Output:** subject-level annualized percent volume change, grouped by split and diagnosis.\n"
    "- **Calculation:** `(last - first) / first / follow-up years`."
))
fig = ctx.plot_ground_truth_start_end_rates()
fig.show()
display(Markdown(
    "After this graph: this is a compact endpoint summary. "
    "The previous curve uses all observed visits, while this box plot uses only the first and last scan per subject."
))


### Ground-truth relative aging curves
- **Input:** all QC-large ground-truth scan volumes in `train`, `val`, and `test`.
- **Output:** mean relative hippocampus volume change from each subject's own baseline, shown separately for `CN` and `AD`.
- **Calculation:** for each subject, `relative change = 100 * (current volume - baseline volume) / baseline volume`.

After this graph: more negative values indicate stronger volume loss from baseline. Because the axis is baseline-relative, subjects with different absolute hippocampus sizes are directly comparable.

### Ground-truth annualized start-to-end change
- **Input:** the first and last observed scan for each subject.
- **Output:** subject-level annualized percent volume change, grouped by split and diagnosis.
- **Calculation:** `(last - first) / first / follow-up years`.

After this graph: this is a compact endpoint summary. The previous curve uses all observed visits, while this box plot uses only the first and last scan per subject.

## Future-pair reconstruction metrics

These are mesh metrics on observed future pairs. Distances stay in millimeters.
Volume quantities are displayed in cubic centimeters and surface-area quantities in square centimeters
so the magnitudes read like hippocampus-scale anatomy instead of raw millimeter powers.


In [3]:
display(Markdown(
    "### Observed future-pair reconstruction error\n"
    "- **Input:** observed source-to-future scan pairs shared by the four comparison models.\n"
    "- **Models:** BrainODE PCA150, PCA150 cocycle flow, SIREN cocycle flow, and SIREN latent ODE.\n"
    "- **Output:** geometric and anatomical endpoint errors after forecasting the future shape. Distances stay in `mm`; volume and surface-area errors are shown in `cm^3` and `cm^2`."
))
fig = ctx.plot_reconstruction_metric_grid()
fig.show()

summary = ctx.model_pair_summary().sort_values(["model_label", "transport_label"]).reset_index(drop=True)
display(summary)
display(Markdown(
    "After this graph: lower bars are better. "
    "ASSD and HD95 summarize surface mismatch; the volume and surface-area panels summarize endpoint anatomy error."
))


### Observed future-pair reconstruction error
- **Input:** observed source-to-future scan pairs shared by the four comparison models.
- **Models:** BrainODE PCA150, PCA150 cocycle flow, SIREN cocycle flow, and SIREN latent ODE.
- **Output:** geometric and anatomical endpoint errors after forecasting the future shape. Distances stay in `mm`; volume and surface-area errors are shown in `cm^3` and `cm^2`.

,model,model_label,transport_method,transport_label,rows,assd_mm,hd95_mm,volume_abs_error_cm3,surface_area_abs_error_cm2,volume_relative_error_pct,surface_area_relative_error_pct
0,qc_brainode_pca150,BrainODE PCA150,brainode_endpoint,endpoint,616,0.397266,0.953809,0.148665,0.532132,5.244451,3.413010
1,pca150_direct_cocycle_flow,PCA150 cocycle flow,composed_observed,composed,4685,0.366449,0.870374,0.135633,0.459840,4.541547,2.872185
2,pca150_direct_cocycle_flow,PCA150 cocycle flow,direct,direct,4685,0.366632,0.868658,0.134613,0.463901,4.460371,2.882153
3,qc_siren_drop_bad_min2,SIREN cocycle flow,composed,composed,616,0.524387,1.190346,0.165503,1.183031,5.824892,7.578552
4,qc_siren_drop_bad_min2,SIREN cocycle flow,direct,direct,616,0.523028,1.188655,0.164847,1.139687,5.799675,7.307718
5,qc_siren_latent_ode,SIREN latent ODE,composed,composed,616,0.519465,1.182097,0.169911,1.158225,5.971341,7.654797
6,qc_siren_latent_ode,SIREN latent ODE,direct,direct,616,0.519391,1.181976,0.169914,1.158108,5.971451,7.654050


After this graph: lower bars are better. ASSD and HD95 summarize surface mismatch; the volume and surface-area panels summarize endpoint anatomy error.

## Cached train/val/test model volume trends

This section uses already-computed observed-age trend tables only when they already exist in the repo.
If a model has no matching cached QC-large split-wide trend table, it is intentionally omitted here.


In [4]:
trend_models = ctx.available_cached_split_trend_models()
if not trend_models:
    display(Markdown("No cached split-wide QC-large model trend tables were found for notebook 1."))
else:
    display(Markdown(
        "The next figures use existing rows from `selected_volume_trends.csv`. "
        "They are not recomputed inside this notebook."
    ))
    for model in trend_models:
        summary = ctx.cached_split_volume_trend_summary(model)
        transport_label = summary["transport"].iloc[0] if not summary.empty else "cached transport"
        display(Markdown(
            f"### {lv.MODEL_LABELS[model]} split-wise CN vs AD trend\n"
            f"- **Input:** cached QC-large observed-age trend rows for `{lv.MODEL_LABELS[model]}`.\n"
            f"- **Transport used:** `{transport_label}`.\n"
            "- **Output:** mean relative volume change from baseline across subjects, separated by split and diagnosis.\n"
            "- **Calculation:** both the observed and predicted curves are normalized by the subject's observed baseline volume."
        ))
        fig = ctx.plot_cached_split_volume_trends(model)
        fig.show()
        display(summary)
        display(Markdown(
            "After this graph: solid lines are observed ground-truth trends and dashed lines are model-predicted trends on the same baseline-relative scale."
        ))


The next figures use existing rows from `selected_volume_trends.csv`. They are not recomputed inside this notebook.

### PCA150 cocycle flow split-wise CN vs AD trend
- **Input:** cached QC-large observed-age trend rows for `PCA150 cocycle flow`.
- **Transport used:** `direct from baseline`.
- **Output:** mean relative volume change from baseline across subjects, separated by split and diagnosis.
- **Calculation:** both the observed and predicted curves are normalized by the subject's observed baseline volume.

,model,transport,split,diagnosis,subjects,scans,median_followup_years,max_followup_years
0,PCA150 cocycle flow,direct from baseline,train,CN,25,236,3.117043,10.255989
1,PCA150 cocycle flow,direct from baseline,train,AD,25,101,0.958244,2.992470
2,PCA150 cocycle flow,direct from baseline,val,CN,25,146,1.330597,10.017792
3,PCA150 cocycle flow,direct from baseline,val,AD,24,72,0.505135,2.072556
4,PCA150 cocycle flow,direct from baseline,test,AD,21,65,0.517452,2.140999
5,PCA150 cocycle flow,direct from baseline,test,CN,25,141,1.245720,10.195763


After this graph: solid lines are observed ground-truth trends and dashed lines are model-predicted trends on the same baseline-relative scale.

### SIREN cocycle flow split-wise CN vs AD trend
- **Input:** cached QC-large observed-age trend rows for `SIREN cocycle flow`.
- **Transport used:** `direct from baseline`.
- **Output:** mean relative volume change from baseline across subjects, separated by split and diagnosis.
- **Calculation:** both the observed and predicted curves are normalized by the subject's observed baseline volume.

,model,transport,split,diagnosis,subjects,scans,median_followup_years,max_followup_years
0,SIREN cocycle flow,direct from baseline,train,AD,13,53,0.960986,2.992471
1,SIREN cocycle flow,direct from baseline,train,CN,13,124,3.990418,10.255989
2,SIREN cocycle flow,direct from baseline,val,CN,15,110,2.027379,10.017796
3,SIREN cocycle flow,direct from baseline,val,AD,15,52,0.532512,2.072553
4,SIREN cocycle flow,direct from baseline,test,AD,15,53,0.517454,2.140999
5,SIREN cocycle flow,direct from baseline,test,CN,15,103,2.061602,10.195756


After this graph: solid lines are observed ground-truth trends and dashed lines are model-predicted trends on the same baseline-relative scale.

## Instantaneous latent velocity

This section evaluates the local latent vector field at every observed scan. It does not decode meshes.

For cocycle flow models, the displayed velocity is the diagonal generator:

`G(z, t, t, c) / age_range_years`

For ODE models, the displayed velocity is the ODE vector field:

`f(z, t, c) / age_range_years`

The observed reference velocity is estimated from neighboring real scans in the same latent space.
PCA and SIREN latent speeds are not compared as raw units; the main plots use train-standardized
latent speed so each model is judged against its own representation scale.


In [5]:
display(Markdown(
    "### Build or load instantaneous velocity cache\n"
    "- **Input:** all train/val/test observed scan latents for BrainODE PCA150, PCA150 cocycle flow, SIREN cocycle flow, and SIREN latent ODE.\n"
    "- **Output:** per-scan instantaneous model velocity, observed scan-to-scan latent velocity, model-vs-real error, and CN-vs-AD condition gap.\n"
    "- **Calculation:** no mesh is generated. The cache is reused on later notebook runs unless the support script version changes."
))
velocity_tables = ctx.build_instantaneous_velocity_tables()
display(velocity_tables["summary"].head(18))
display(Markdown(
    f"Loaded `{len(velocity_tables['per_scan'])}` per-scan/condition velocity rows and "
    f"`{len(velocity_tables['condition_gap'])}` CN-vs-AD condition-gap rows."
))

display(Markdown(
    "### Model velocity magnitude vs real local velocity\n"
    "- **Input:** observed-condition rows only.\n"
    "- **Output:** bars show the model's median instantaneous latent speed; `x` markers show the median real scan-to-scan latent speed.\n"
    "- **Calculation:** speeds are divided by the training latent standard deviation component-by-component, then summarized as an L2 norm per year."
))
fig = ctx.plot_instantaneous_velocity_model_vs_real()
fig.show()
display(Markdown(
    "After this graph: if a model bar is far below the real marker for AD, the model is underestimating AD progression speed even if its future meshes look plausible."
))

display(Markdown(
    "### Velocity error and direction agreement\n"
    "- **Input:** observed-condition rows only.\n"
    "- **Output:** top row = median train-standardized velocity error; bottom row = median cosine similarity between model velocity and observed local velocity.\n"
    "- **Calculation:** cosine near `1` means the model moves in the same latent direction as the observed scan-to-scan change; near `0` means weak directional agreement."
))
fig = ctx.plot_instantaneous_velocity_alignment()
fig.show()
display(Markdown(
    "After this graph: this separates speed magnitude from direction. A model can have plausible speed but still move in the wrong latent direction."
))

display(Markdown(
    "### CN-vs-AD instantaneous condition gap\n"
    "- **Input:** the same source scan evaluated twice: once with CN condition and once with AD condition.\n"
    "- **Output:** `AD-conditioned speed - CN-conditioned speed`, grouped by source diagnosis and split.\n"
    "- **Calculation:** positive values mean the AD condition makes the instantaneous vector field faster than the CN condition for the same scan."
))
fig = ctx.plot_instantaneous_velocity_condition_gap()
fig.show()
display(ctx.instantaneous_velocity_condition_gap_summary())
display(Markdown(
    "After this graph: this is the direct local test of whether the condition label changes the velocity field in the expected disease direction."
))

display(Markdown(
    "### Velocity by age bin\n"
    "- **Input:** observed-condition rows only.\n"
    "- **Output:** median instantaneous model speed and median observed local speed by age bin, separated by CN and AD.\n"
    "- **Calculation:** this checks whether the learned vector field changes with age rather than only with diagnosis."
))
fig = ctx.plot_instantaneous_velocity_age_trend()
fig.show()
display(Markdown(
    "After this graph: a clinically useful disease-aging model should show both age-dependent behavior and stronger AD velocity than CN where the real data shows that pattern."
))


### Build or load instantaneous velocity cache
- **Input:** all train/val/test observed scan latents for BrainODE PCA150, PCA150 cocycle flow, SIREN cocycle flow, and SIREN latent ODE.
- **Output:** per-scan instantaneous model velocity, observed scan-to-scan latent velocity, model-vs-real error, and CN-vs-AD condition gap.
- **Calculation:** no mesh is generated. The cache is reused on later notebook runs unless the support script version changes.

,dataset,model,model_label,model_priority,latent_family,generator_kind,split,diagnosis,condition_eval,rows,...,model_velocity_z_l2_median,velocity_l2_error_mean,velocity_l2_error_median,velocity_z_l2_error_mean,velocity_z_l2_error_median,velocity_cosine_mean,velocity_cosine_median,model_to_real_speed_ratio_median,split_priority,condition_priority
0,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,train,AD,observed,496,...,0.292383,2.045445,1.817796,17.281141,14.665344,0.154445,0.164972,0.050795,0,0
1,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,train,AD,CN,496,...,0.273100,2.046175,1.819147,17.280924,14.667375,0.154053,0.165100,0.047043,0,1
2,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,train,AD,AD,496,...,0.352013,2.043767,1.813253,17.282120,14.660766,0.154980,0.164128,0.060131,0,2
3,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,train,AD,observed,496,...,0.975715,2.046149,1.834415,17.254508,14.749336,0.118377,0.160087,0.080377,0,0
4,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,train,AD,CN,496,...,0.926795,2.055132,1.831418,17.259705,14.742296,0.057969,0.104590,0.071536,0,1
5,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,train,AD,AD,496,...,0.975715,2.046149,1.834415,17.254508,14.749336,0.118377,0.160087,0.080377,0,2
6,qc_large,qc_siren_drop_bad_min2,SIREN cocycle flow,2,siren256,cocycle_diagonal_generator,train,AD,observed,496,...,2.276088,1.349831,1.102713,25.284421,20.709735,-0.000038,-0.000238,0.109170,0,0
7,qc_large,qc_siren_drop_bad_min2,SIREN cocycle flow,2,siren256,cocycle_diagonal_generator,train,AD,CN,496,...,1.450282,1.345745,1.098314,25.209900,20.629322,-0.000298,-0.001005,0.065816,0,1
8,qc_large,qc_siren_drop_bad_min2,SIREN cocycle flow,2,siren256,cocycle_diagonal_generator,train,AD,AD,496,...,2.276088,1.349831,1.102713,25.284421,20.709735,-0.000038,-0.000238,0.109170,0,2
9,qc_large,qc_siren_latent_ode,SIREN latent ODE,3,siren256,ode_vector_field,train,AD,observed,496,...,8.316171,1.430597,1.185882,26.768481,22.176803,-0.000177,-0.002183,0.401074,0,0


Loaded `26748` per-scan/condition velocity rows and `8916` CN-vs-AD condition-gap rows.

### Model velocity magnitude vs real local velocity
- **Input:** observed-condition rows only.
- **Output:** bars show the model's median instantaneous latent speed; `x` markers show the median real scan-to-scan latent speed.
- **Calculation:** speeds are divided by the training latent standard deviation component-by-component, then summarized as an L2 norm per year.

After this graph: if a model bar is far below the real marker for AD, the model is underestimating AD progression speed even if its future meshes look plausible.

...

### CN-vs-AD instantaneous condition gap
- **Input:** the same source scan evaluated twice: once with CN condition and once with AD condition.
- **Output:** `AD-conditioned speed - CN-conditioned speed`, grouped by source diagnosis and split.
- **Calculation:** positive values mean the AD condition makes the instantaneous vector field faster than the CN condition for the same scan.

,dataset,model,model_label,model_priority,latent_family,generator_kind,split,diagnosis,rows,subjects,scans,ad_minus_cn_velocity_z_mean,ad_minus_cn_velocity_z_median,ad_minus_cn_velocity_l2_mean,ad_minus_cn_velocity_l2_median,ad_over_cn_velocity_z_ratio_median,split_priority
0,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,train,AD,496,160,496,0.078904,0.078907,0.023773,0.023774,1.288967,0
1,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,train,AD,496,160,496,0.030728,0.027794,0.014682,0.014294,1.029370,0
2,qc_large,qc_siren_drop_bad_min2,SIREN cocycle flow,2,siren256,cocycle_diagonal_generator,train,AD,496,160,496,0.825529,0.826462,0.044644,0.044694,1.569620,0
3,qc_large,qc_siren_latent_ode,SIREN latent ODE,3,siren256,ode_vector_field,train,AD,496,160,496,4.267363,4.275165,0.230134,0.230512,2.064842,0
4,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,train,CN,1295,261,1295,0.079127,0.079142,0.023777,0.023778,1.293807,0
5,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,train,CN,1295,261,1295,0.070553,0.070827,0.031860,0.034228,1.073552,0
6,qc_large,qc_siren_drop_bad_min2,SIREN cocycle flow,2,siren256,cocycle_diagonal_generator,train,CN,1295,261,1295,0.823409,0.823228,0.044532,0.044521,1.589751,0
7,qc_large,qc_siren_latent_ode,SIREN latent ODE,3,siren256,ode_vector_field,train,CN,1295,261,1295,4.240378,4.239316,0.228677,0.228619,2.163669,0
8,qc_large,qc_brainode_pca150,BrainODE PCA150,0,pca150,ode_vector_field,val,AD,72,24,72,0.078948,0.078970,0.023774,0.023774,1.290276,1
9,qc_large,pca150_direct_cocycle_flow,PCA150 cocycle flow,1,pca150,cocycle_diagonal_generator,val,AD,72,24,72,0.036493,0.036268,0.017338,0.021366,1.040002,1


After this graph: this is the direct local test of whether the condition label changes the velocity field in the expected disease direction.

### Velocity by age bin
- **Input:** observed-condition rows only.
- **Output:** median instantaneous model speed and median observed local speed by age bin, separated by CN and AD.
- **Calculation:** this checks whether the learned vector field changes with age rather than only with diagnosis.

After this graph: a clinically useful disease-aging model should show both age-dependent behavior and stronger AD velocity than CN where the real data shows that pattern.

## Representative observed-pair cases

Cases are chosen split-by-split and diagnosis-by-diagnosis from pairs that exist for all four models.
Preference is given to longer gaps, then lower average future-pair error.


In [6]:
cases = ctx.representative_cases()
case_table = pd.DataFrame(
    [
        {
            "split": case.split,
            "diagnosis": case.diagnosis,
            "subject_id": case.subject_id,
            "source_scan_id": case.source_scan_id,
            "target_scan_id": case.target_scan_id,
        }
        for case in cases
    ]
)
display(case_table)
display(Markdown(
    "Each row defines one source-to-target case used below. "
    "The first case-study section shows only the selected target prediction per model; the next case-study section expands that into predictions at each observed follow-up age between source and target."
))


,split,diagnosis,subject_id,source_scan_id,target_scan_id
0,train,CN,31,31_bl_left,31_m120_left
1,train,AD,995,995_bl_left,995_m36_left
2,val,CN,677,677_bl_left,677_m120_left
3,val,AD,850,850_bl_left,850_m24_left
4,test,CN,113,113_bl_left,113_m108_left
5,test,AD,1339,1339_bl_left,1339_m24_left


Each row defines one source-to-target case used below. The first case-study section shows only the selected target prediction per model; the next case-study section expands that into predictions at each observed follow-up age between source and target.

## Volume history around the selected future-pair cases

Each panel shows the real subject trajectory and the model-predicted target volume at the selected future age.
Volumes are displayed in cubic centimeters.


In [7]:
for case in cases:
    display(Markdown(f"### {case.split.upper()} {case.diagnosis} | subject {case.subject_id}"))
    display(ctx.case_overview_table(case))
    display(Markdown(
        "- **Input:** the full observed subject history plus the selected source-to-target pair for this case.\n"
        "- **Output:** black line = all observed scan volumes for the subject; colored diamonds = each model's prediction only at the selected target age.\n"
        "- **Important:** intermediate observed ages before the target are shown only on the black ground-truth line in this plot."
    ))
    fig = ctx.plot_case_subject_history(case)
    fig.show()
    display(Markdown(
        "After this graph: this is a target-only endpoint comparison. "
        "The next section fills in the missing intermediate predictions before the target."
    ))


### TRAIN CN | subject 31

,model,chosen_transport,gap_years,chamfer_l2_squared,assd_mm,hd95_mm,volume_abs_error_cm3,surface_area_abs_error_cm2
0,BrainODE PCA150,endpoint,10.255989,0.002070,0.540511,1.487205,0.559828,1.883015
1,PCA150 cocycle flow,composed,10.255989,0.000857,0.376803,0.988132,0.062958,0.054143
2,SIREN cocycle flow,direct,10.255988,0.002718,0.675967,1.649827,0.535662,5.268731
3,SIREN latent ODE,direct,10.255988,0.003244,0.775844,1.719510,0.515956,1.458915


- **Input:** the full observed subject history plus the selected source-to-target pair for this case.
- **Output:** black line = all observed scan volumes for the subject; colored diamonds = each model's prediction only at the selected target age.
- **Important:** intermediate observed ages before the target are shown only on the black ground-truth line in this plot.

After this graph: this is a target-only endpoint comparison. The next section fills in the missing intermediate predictions before the target.

### TRAIN AD | subject 995

...

After this graph: this is a target-only endpoint comparison. The next section fills in the missing intermediate predictions before the target.

### TEST AD | subject 1339

,model,chosen_transport,gap_years,chamfer_l2_squared,assd_mm,hd95_mm,volume_abs_error_cm3,surface_area_abs_error_cm2
0,BrainODE PCA150,endpoint,2.140999,0.001244,0.471948,1.157105,0.229941,0.276845
1,PCA150 cocycle flow,direct,2.140999,0.001185,0.462451,1.133379,0.238488,0.271595
2,SIREN cocycle flow,direct,2.141001,0.001329,0.513087,1.128257,0.318092,0.895384
3,SIREN latent ODE,composed,2.141001,0.001373,0.509474,1.174647,0.226899,3.334334


- **Input:** the full observed subject history plus the selected source-to-target pair for this case.
- **Output:** black line = all observed scan volumes for the subject; colored diamonds = each model's prediction only at the selected target age.
- **Important:** intermediate observed ages before the target are shown only on the black ground-truth line in this plot.

After this graph: this is a target-only endpoint comparison. The next section fills in the missing intermediate predictions before the target.

## Per-model predictions at every observed follow-up age between source and target

This section addresses the gap in the simpler history plots above: if a subject has observed follow-up scans
before the selected target, each model is now evaluated at those observed ages as well, using the same source scan as input.


In [8]:
for case in cases:
    display(Markdown(f"## {case.split.upper()} {case.diagnosis} | subject {case.subject_id} | source-to-observed follow-up forecasts"))
    for model in ctx.available_models():
        fig, table = ctx.plot_case_model_followup_forecast(case, model)
        first = table.iloc[0]
        transport_label = str(table['transport_label'].dropna().iloc[0]) if table['transport_label'].notna().any() else 'endpoint'
        display(Markdown(
            f"### {lv.MODEL_LABELS[model]}\n"
            f"- **Input:** source scan `{first['source_scan_id']}` at age `{first['source_age_years']:.2f}` years with diagnosis `{first['diagnosis']}`.\n"
            f"- **Output:** predicted hippocampus volume at each observed follow-up age from the source age up to the selected target age `{first['target_age_years']:.2f}`.\n"
            f"- **Transport / mode:** `{transport_label}`.\n"
            "- **Calculation:** ground truth uses the physical mesh volume when available; the model curve uses the volume of the decoded forecast mesh."
        ))
        fig.show()
        display(table)
        ok = table.loc[table['status'].astype(str).eq('ok') & table['predicted_volume_cm3'].notna()].copy()
        mae = float(ok['abs_error_cm3'].mean()) if not ok.empty else float('nan')
        skipped = int(table['status'].astype(str).ne('ok').sum())
        display(Markdown(
            f"After this graph: mean absolute error across the shown observed follow-up ages is `{mae:.3f} cm^3` "
            f"for the successful predictions. Skipped ages due to failed mesh decode: `{skipped}`."
        ))


## TRAIN CN | subject 31 | source-to-observed follow-up forecasts

### BrainODE PCA150
- **Input:** source scan `31_bl_left` at age `77.70` years with diagnosis `CN`.
- **Output:** predicted hippocampus volume at each observed follow-up age from the source age up to the selected target age `87.96`.
- **Transport / mode:** `endpoint`.
- **Calculation:** ground truth uses the physical mesh volume when available; the model curve uses the volume of the decoded forecast mesh.

,split,diagnosis,subject_id,model_label,transport_label,source_scan_id,source_age_years,target_scan_id,target_age_years,future_scan_id,future_age_years,is_target_scan,ground_truth_volume_cm3,predicted_volume_cm3,abs_error_cm3,status,error
0,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_bl_left,77.700000,False,3.725230,3.726255,0.001024,ok,NaN
1,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m06_left,78.181862,False,3.842658,3.714657,0.128001,ok,NaN
2,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m12_left,78.715743,False,4.128782,3.701755,0.427027,ok,NaN
3,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m24_left,79.756126,False,3.372957,3.676459,0.303503,ok,NaN
4,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m36_left,80.807461,False,3.501588,3.650694,0.149106,ok,NaN
5,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m48_left,81.694524,False,3.532291,3.628797,0.096506,ok,NaN
6,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m60_left,82.803354,False,3.402475,3.601226,0.198752,ok,NaN
7,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m72_left,83.687680,False,3.529717,3.579082,0.049364,ok,NaN
8,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m84_left,84.599384,False,3.364198,3.556109,0.191910,ok,NaN
9,train,CN,31,BrainODE PCA150,endpoint,31_bl_left,77.699997,31_m120_left,87.955986,31_m96_left,85.680835,False,3.346060,3.528673,0.182613,ok,NaN


After this graph: mean absolute error across the shown observed follow-up ages is `0.173 cm^3` for the successful predictions. Skipped ages due to failed mesh decode: `0`.

### PCA150 cocycle flow
- **Input:** source scan `31_bl_left` at age `77.70` years with diagnosis `CN`.
- **Output:** predicted hippocampus volume at each observed follow-up age from the source age up to the selected target age `87.96`.
- **Transport / mode:** `composed`.
- **Calculation:** ground truth uses the physical mesh volume when available; the model curve uses the volume of the decoded forecast mesh.

...

,split,diagnosis,subject_id,model_label,transport_label,source_scan_id,source_age_years,target_scan_id,target_age_years,future_scan_id,future_age_years,is_target_scan,ground_truth_volume_cm3,predicted_volume_cm3,abs_error_cm3,status,error
0,test,AD,1339,SIREN cocycle flow,direct,1339_bl_left,79.5,1339_m24_left,81.640999,1339_bl_left,79.500000,False,2.133729,2.129237,0.004492,ok,NaN
1,test,AD,1339,SIREN cocycle flow,direct,1339_bl_left,79.5,1339_m24_left,81.640999,1339_m06_left,80.039357,False,1.981510,2.138708,0.157199,ok,NaN
2,test,AD,1339,SIREN cocycle flow,direct,1339_bl_left,79.5,1339_m24_left,81.640999,1339_m12_left,80.567762,False,2.137270,2.148430,0.011160,ok,NaN


After this graph: mean absolute error across the shown observed follow-up ages is `0.058 cm^3` for the successful predictions. Skipped ages due to failed mesh decode: `0`.

### SIREN latent ODE
- **Input:** source scan `1339_bl_left` at age `79.50` years with diagnosis `AD`.
- **Output:** predicted hippocampus volume at each observed follow-up age from the source age up to the selected target age `81.64`.
- **Transport / mode:** `composed`.
- **Calculation:** ground truth uses the physical mesh volume when available; the model curve uses the volume of the decoded forecast mesh.

,split,diagnosis,subject_id,model_label,transport_label,source_scan_id,source_age_years,target_scan_id,target_age_years,future_scan_id,future_age_years,is_target_scan,ground_truth_volume_cm3,predicted_volume_cm3,abs_error_cm3,status,error
0,test,AD,1339,SIREN latent ODE,composed,1339_bl_left,79.5,1339_m24_left,81.640999,1339_bl_left,79.500000,False,2.133729,2.129237,0.004492,ok,NaN
1,test,AD,1339,SIREN latent ODE,composed,1339_bl_left,79.5,1339_m24_left,81.640999,1339_m06_left,80.039357,False,1.981510,2.190362,0.208853,ok,NaN
2,test,AD,1339,SIREN latent ODE,composed,1339_bl_left,79.5,1339_m24_left,81.640999,1339_m12_left,80.567762,False,2.137270,2.192824,0.055554,ok,NaN


After this graph: mean absolute error across the shown observed follow-up ages is `0.090 cm^3` for the successful predictions. Skipped ages due to failed mesh decode: `0`.

## Conditioned future volume forecasts from a single shape

These forecasts start from the first observed scan of the selected anchor subject.
Solid lines use the CN condition. Dashed lines use the AD condition.
Larger markers indicate prediction points at the exact observed follow-up ages for that subject.
The grey region begins after the last observed scan for the chosen anchor subject.
Volumes are displayed in cubic centimeters.


In [9]:
display(Markdown(
    "### Test CN anchor forecast\n"
    "- **Input:** one CN source scan from the test split, used as a fixed starting shape.\n"
    "- **Output:** future predicted volume under both the `CN` and `AD` condition labels.\n"
    "- **Calculation:** black is observed truth; solid colored lines are CN-conditioned forecasts; dashed colored lines are AD-conditioned forecasts."
))
fig = ctx.plot_anchor_forecasts(split="test", diagnosis="CN", transport="direct")
fig.show()
display(Markdown(
    "After this graph: the difference between the solid and dashed curves shows how much the diagnosis condition changes the future rollout while the source anatomy stays fixed."
))

display(Markdown(
    "### Test AD anchor forecast\n"
    "- **Input:** one AD source scan from the test split, used as a fixed starting shape.\n"
    "- **Output:** future predicted volume under both the `AD` and `CN` condition labels.\n"
    "- **Calculation:** the same source shape is rolled forward to observed and out-of-distribution ages, with the grey region starting after the subject's last observed scan."
))
fig = ctx.plot_anchor_forecasts(split="test", diagnosis="AD", transport="direct")
fig.show()
display(Markdown(
    "After this graph: this is a conditioning sensitivity test. "
    "It does not prove disentanglement, but it does show the direction and size of the label-dependent forecast change."
))


### Test CN anchor forecast
- **Input:** one CN source scan from the test split, used as a fixed starting shape.
- **Output:** future predicted volume under both the `CN` and `AD` condition labels.
- **Calculation:** black is observed truth; solid colored lines are CN-conditioned forecasts; dashed colored lines are AD-conditioned forecasts.

After this graph: the difference between the solid and dashed curves shows how much the diagnosis condition changes the future rollout while the source anatomy stays fixed.

### Test AD anchor forecast
- **Input:** one AD source scan from the test split, used as a fixed starting shape.
- **Output:** future predicted volume under both the `AD` and `CN` condition labels.
- **Calculation:** the same source shape is rolled forward to observed and out-of-distribution ages, with the grey region starting after the subject's last observed scan.

After this graph: this is a conditioning sensitivity test. It does not prove disentanglement, but it does show the direction and size of the label-dependent forecast change.

## Disease-conditioned gap

This is not disentanglement. It is the forecast separation induced by swapping the conditioning label
while keeping the source shape fixed.
Volume gaps are displayed in cubic centimeters.


In [10]:
display(Markdown(
    "### Test CN anchor disease-conditioned gap\n"
    "- **Input:** the CN anchor forecast from the previous section.\n"
    "- **Output:** `AD-conditioned predicted volume - CN-conditioned predicted volume` at each future age.\n"
    "- **Calculation:** each point subtracts two forecasts made from the same source shape."
))
fig = ctx.plot_anchor_condition_gap(split="test", diagnosis="CN", transport="direct")
fig.show()
display(Markdown(
    "After this graph: values below zero mean the AD-conditioned rollout predicts a smaller hippocampus than the CN-conditioned rollout."
))

display(Markdown(
    "### Test AD anchor disease-conditioned gap\n"
    "- **Input:** the AD anchor forecast from the previous section.\n"
    "- **Output:** the same `AD - CN` conditional gap, now starting from an AD source shape.\n"
    "- **Calculation:** the source scan is fixed and only the condition label is swapped."
))
fig = ctx.plot_anchor_condition_gap(split="test", diagnosis="AD", transport="direct")
fig.show()
display(Markdown(
    "After this graph: this quantifies label sensitivity on a shared source shape, rather than measuring reconstruction against ground truth."
))


### Test CN anchor disease-conditioned gap
- **Input:** the CN anchor forecast from the previous section.
- **Output:** `AD-conditioned predicted volume - CN-conditioned predicted volume` at each future age.
- **Calculation:** each point subtracts two forecasts made from the same source shape.

After this graph: values below zero mean the AD-conditioned rollout predicts a smaller hippocampus than the CN-conditioned rollout.

### Test AD anchor disease-conditioned gap
- **Input:** the AD anchor forecast from the previous section.
- **Output:** the same `AD - CN` conditional gap, now starting from an AD source shape.
- **Calculation:** the source scan is fixed and only the condition label is swapped.

After this graph: this quantifies label sensitivity on a shared source shape, rather than measuring reconstruction against ground truth.